# 04 — Robustness Checks (CRISP-DM Evaluation)
The main model answers: *which characteristics separate repaid from charged-off loans?* These two checks ask whether that answer can be trusted.

| Check | Question | Method |
|---|---|---|
| **A. Vintage / censoring** | Are findings (especially loan term) an artifact of modeling only *resolved* loans? | Refit on samples where selection is milder: FY2010–2019 loans, and loans whose term has ended |
| **B. Time-based** | Do patterns learned from the past hold on later loans? | Train FY2010–2015, test FY2016–2019 |

**Design rules**
- Uses **training rows only** (`split == "train"`). The sealed test set is reserved for the final main model.
- Same simple model in every comparison (class-weighted logistic regression), so differences come from the *data*, not the model. Preprocessing is refit inside every fit — no leakage.
- Cross-validation is **grouped by `LocationID`**, matching the main split.
- Metrics: ROC-AUC (ranking quality) and PR-AUC (compare to the default rate, which is what a useless model scores).

## 0. Setup

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"
SEED = 42
RUN_GBM = True          # set False to skip the gradient-boosting comparison in Check B (faster)
pd.set_option("display.max_columns", 40, "display.width", 150, "display.float_format", "{:,.3f}".format)

NUM = ["log_gross_approval", "guarantee_pct", "TermInMonths", "InitialInterestRate",
       "log_jobs_supported", "log_approval_lag_days"]
CAT = ["naics_sector", "BusinessType", "BusinessAge", "ProjectState", "ProcessingMethod",
       "FixedorVariableInterestInd", "RevolverStatus", "CollateralInd", "is_franchise",
       "lender_type", "lender_volume_tier"]

data = pd.read_csv(PROC / "analytic_table.csv", keep_default_na=True, low_memory=False)
train = data[data["split"] == "train"].reset_index(drop=True)       # sealed test rows are never touched here
print(f"training rows: {len(train):,} | default rate {train['default'].mean():.2%} | FY {train['approval_fy'].min()}-{train['approval_fy'].max()}")

training rows: 403,162 | default rate 8.14% | FY 2010-2026


### Helpers
Same preprocessing as `02_data_preparation` (median impute + indicator + scale; "Missing" + one-hot with rare pooling), wrapped in a function so it is **refit on whatever sample is being analyzed**.

In [2]:
def make_pre(min_freq=500):
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale", StandardScaler())]), NUM),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                   min_frequency=min_freq, sparse_output=True))]), CAT)])

def frame(d):
    x = d[NUM + CAT].copy()
    for c in CAT: x[c] = x[c].astype(object).where(x[c].notna(), np.nan)
    for c in NUM: x[c] = x[c].astype(float)
    return x

def lr():  return LogisticRegression(class_weight="balanced", max_iter=500, C=1.0)
def gbm(): return HistGradientBoostingClassifier(class_weight="balanced", max_iter=200, learning_rate=0.1, random_state=SEED)

def groups_of(d):
    g = d["LocationID"].copy()
    g[g.isna()] = -(np.arange(g.isna().sum()) + 1)
    return g.to_numpy()

def cv_scores(d, model_fn, n_splits=5):
    """Grouped, stratified CV on one sample. Preprocessor is refit in every fold."""
    X, y, g = frame(d), d["default"].to_numpy(), groups_of(d)
    rows = []
    for k, (a, b) in enumerate(StratifiedGroupKFold(n_splits, shuffle=True, random_state=SEED).split(X, y, g)):
        pipe = Pipeline([("pre", make_pre()), ("clf", model_fn())]).fit(X.iloc[a], y[a])
        p = pipe.predict_proba(X.iloc[b])[:, 1]
        rows.append({"fold": k, "roc_auc": roc_auc_score(y[b], p), "pr_auc": average_precision_score(y[b], p)})
    return pd.DataFrame(rows)

def fit_lr(d):
    """Fit on a whole sample; return coefficients (by feature name) and the fitted pipeline."""
    pipe = Pipeline([("pre", make_pre()), ("clf", lr())]).fit(frame(d), d["default"].to_numpy())
    coefs = pd.Series(pipe["clf"].coef_[0], index=pipe["pre"].get_feature_names_out())
    return coefs, pipe

def term_odds_ratio_per_year(pipe, coefs):
    """Odds multiplier for +12 months of term, holding other features fixed (undoes the scaling)."""
    sd = pipe["pre"].named_transformers_["num"]["scale"].scale_[NUM.index("TermInMonths")]
    return float(np.exp(coefs["num__TermInMonths"] / sd * 12))

---
# Check A — Vintage / censoring
**The concern.** Only resolved loans are modeled. In FY2020+ just 19–50% of disbursed loans have resolved, and the ones that did are mostly short loans and early outcomes; older vintages are 75–99% resolved. If `TermInMonths` looks protective only because long recent loans haven't had time to default, the effect is partly an artifact.

**Three samples (all from the training rows)**

| Sample | Why |
|---|---|
| **All resolved** | The main-model sample |
| **FY2010–2019** | Nearly fully resolved, so little selection bias |
| **Matured only** | Loans whose scheduled term has already ended — their outcomes are essentially complete |

In [3]:
samples = {
    "All resolved":  train,
    "FY2010-2019":   train[train["approval_fy"] <= 2019],
    "Matured only":  train[train["matured"] == 1],
}
for k, d in samples.items():
    print(f"{k:14s} {len(d):>8,} loans | {d['default'].sum():>6,} defaults ({d['default'].mean():.1%})")

All resolved    403,162 loans | 32,831 defaults (8.1%)
FY2010-2019     346,227 loans | 27,184 defaults (7.9%)
Matured only    264,753 loans | 25,282 defaults (9.5%)


### A1. Does model performance depend on the sample?

In [4]:
rows = []
for name, d in samples.items():
    s = cv_scores(d, lr)
    rows.append({"sample": name, "loans": len(d), "default_rate": d["default"].mean(),
                 "roc_auc": s["roc_auc"].mean(), "roc_auc_sd": s["roc_auc"].std(),
                 "pr_auc": s["pr_auc"].mean(), "pr_auc_sd": s["pr_auc"].std()})
A1 = pd.DataFrame(rows).set_index("sample")
A1

,loans,default_rate,roc_auc,roc_auc_sd,pr_auc,pr_auc_sd
sample,,,,,,
All resolved,403162,0.081,0.845,0.019,0.402,0.057
FY2010-2019,346227,0.079,0.849,0.010,0.436,0.025
Matured only,264753,0.095,0.884,0.014,0.496,0.024


**How to read it.** `pr_auc` should be compared with `default_rate` (a model with no skill scores about the default rate). If ROC-AUC is similar across samples, the signal is not just a quirk of which loans resolved; if it is much higher/lower in one sample, that sample's outcomes are being driven by something different.

### A2. Does the term effect survive? (the key question)

**Raw view:** default rate by term bucket in each sample, with no modeling.

In [5]:
bins, labels = [0, 60, 84, 120, 1000], ["<=60", "61-84", "85-120", "121+"]
rows = {}
for name, d in samples.items():
    b = pd.cut(d["TermInMonths"], bins, labels=labels)
    rows[name] = d.groupby(b, observed=True)["default"].mean()
A2_raw = pd.DataFrame(rows); A2_raw.index.name = "term (months)"
A2_raw

,All resolved,FY2010-2019,Matured only
term (months),,,
<=60,0.201,0.214,0.205
61-84,0.050,0.045,0.046
85-120,0.072,0.058,0.065
121+,0.024,0.027,0.039


**Modeled view:** odds multiplier for each extra 12 months of term, holding other features fixed (below 1 = longer terms are protective).

In [6]:
fits = {name: fit_lr(d) for name, d in samples.items()}          # (coefs, pipe) per sample
A2_model = pd.Series({n: term_odds_ratio_per_year(p, c) for n, (c, p) in fits.items()}, name="odds x per +12 months of term")
A2_model.to_frame()

,odds x per +12 months of term
All resolved,0.714
FY2010-2019,0.708
Matured only,0.472


**Verdict rule of thumb.**
- Multiplier stays below 1 with a similar size in all three samples → the term effect is **robust**; report it with confidence.
- Multiplier shrinks toward 1 (or flips) in FY2010–2019 / matured samples → the main-sample effect was **partly a selection artifact**; report the smaller number as the credible one.
- Raw view and modeled view disagree → another feature (e.g., loan size or loan type) is carrying the effect.

### A3. Are the other findings stable across samples?

In [7]:
coef_tbl = pd.DataFrame({n: c for n, (c, p) in fits.items()}).dropna()          # features present in every sample
print("Spearman rank correlation of coefficients between samples (1 = same ordering):")
print(coef_tbl.corr(method="spearman").round(2))

top = coef_tbl["All resolved"].abs().sort_values(ascending=False).head(15).index
tbl = coef_tbl.loc[top].copy()
tbl["same_sign_in_all"] = np.sign(tbl).nunique(axis=1).eq(1)
print("\nTop 15 features by |coefficient| in the main sample (standardized/one-hot scale):")
tbl

Spearman rank correlation of coefficients between samples (1 = same ordering):
              All resolved  FY2010-2019  Matured only
All resolved         1.000        0.940         0.910
FY2010-2019          0.940        1.000         0.950
Matured only         0.910        0.950         1.000

Top 15 features by |coefficient| in the main sample (standardized/one-hot scale):


,All resolved,FY2010-2019,Matured only,same_sign_in_all
cat__ProcessingMethod_7awithEWCP,-3.540,-3.407,-5.058,True
cat__BusinessAge_Missing,2.158,2.292,2.096,True
num__TermInMonths,-2.144,-2.191,-1.840,True
cat__ProcessingMethod_infrequent_sklearn,-1.461,-2.861,-0.733,True
cat__ProcessingMethod_7aGeneral,1.372,1.175,2.175,True
cat__ProcessingMethod_CertifiedLendersProgram,1.289,1.031,1.497,True
cat__ProcessingMethod_RuralLoanInitiative,1.075,0.846,1.920,True
cat__ProcessingMethod_CommunityAdvantageInitiative,0.929,0.211,0.601,True
cat__ProcessingMethod_SmallLoanAdvantageInitiative,0.756,0.595,1.610,True
cat__ProcessingMethod_CommunityExpress,0.736,0.602,1.331,True


`same_sign_in_all = False` marks a finding that changes direction depending on the sample — treat those cautiously in the write-up.

---
# Check B — Time-based validation
**The concern.** The main split is random, so a model can learn patterns from 2018 loans and be tested on 2018 loans. A real lender scores *future* loans.

**Design.** Train on FY2010–2015 and test on FY2016–2019 — all older, mostly resolved vintages. FY2020+ is excluded as a test set because most of those loans are still unresolved. Only training rows are used.

**Reference point.** To separate "the future is harder" from "this era is just harder", we also report random grouped CV *within* FY2016–2019. The gap between the two is the cost of predicting forward in time.

In [8]:
past  = train[train["approval_fy"].between(2010, 2015)]
later = train[train["approval_fy"].between(2016, 2019)]
print(f"train (FY2010-2015): {len(past):,} loans, {past['default'].mean():.1%} default")
print(f"test  (FY2016-2019): {len(later):,} loans, {later['default'].mean():.1%} default")

train (FY2010-2015): 209,972 loans, 7.2% default
test  (FY2016-2019): 136,255 loans, 8.8% default


### B1. Forward-in-time performance vs. same-era random CV

In [9]:
def forward_eval(model_fn):
    pipe = Pipeline([("pre", make_pre()), ("clf", model_fn())]).fit(frame(past), past["default"].to_numpy())
    p = pipe.predict_proba(frame(later))[:, 1]
    out = {"ALL 2016-19": (roc_auc_score(later["default"], p), average_precision_score(later["default"], p), later["default"].mean(), len(later))}
    for fy, d in later.groupby("approval_fy"):
        q = p[later["approval_fy"].to_numpy() == fy]
        out[f"FY{fy}"] = (roc_auc_score(d["default"], q), average_precision_score(d["default"], q), d["default"].mean(), len(d))
    return pd.DataFrame(out, index=["roc_auc", "pr_auc", "default_rate", "loans"]).T

B1_lr = forward_eval(lr)
ref = cv_scores(later, lr)
print("Logistic regression, trained on FY2010-2015, tested forward:")
display(B1_lr)
print(f"\nSame-era reference (random grouped CV inside FY2016-2019): ROC-AUC {ref['roc_auc'].mean():.3f}, PR-AUC {ref['pr_auc'].mean():.3f}")
print(f"Cost of predicting forward in time: ROC-AUC {ref['roc_auc'].mean() - B1_lr.loc['ALL 2016-19','roc_auc']:+.3f}, "
      f"PR-AUC {ref['pr_auc'].mean() - B1_lr.loc['ALL 2016-19','pr_auc']:+.3f}")

Logistic regression, trained on FY2010-2015, tested forward:


,roc_auc,pr_auc,default_rate,loans
ALL 2016-19,0.809,0.287,0.088,"136,255.000"
FY2016,0.862,0.441,0.076,"41,140.000"
FY2017,0.867,0.482,0.086,"37,246.000"
FY2018,0.785,0.319,0.098,"32,860.000"
FY2019,0.766,0.308,0.099,"25,009.000"



Same-era reference (random grouped CV inside FY2016-2019): ROC-AUC 0.874, PR-AUC 0.514
Cost of predicting forward in time: ROC-AUC +0.066, PR-AUC +0.227


**How to read it.** A small gap means patterns learned on older loans transfer to newer ones (stable). A large gap, or AUC that falls year by year, means the relationships drift — the model would need regular retraining, which is a useful deployment point for Assignment 3. Compare each year's PR-AUC with its own `default_rate`, since the base rate moves over time.

### B2. Does a more flexible model transfer better? (optional, set `RUN_GBM`)

In [10]:
if RUN_GBM:
    # HistGradientBoosting needs dense input
    def forward_eval_gbm():
        pre = make_pre(); Xp = pre.fit_transform(frame(past)); Xl = pre.transform(frame(later))
        dense = lambda m: (m.toarray() if hasattr(m, "toarray") else np.asarray(m)).astype("float32")
        Xp, Xl = dense(Xp), dense(Xl)
        m = gbm().fit(Xp, past["default"].to_numpy()); p = m.predict_proba(Xl)[:, 1]
        return roc_auc_score(later["default"], p), average_precision_score(later["default"], p)
    g_auc, g_pr = forward_eval_gbm()
    B2 = pd.DataFrame({"roc_auc": [B1_lr.loc["ALL 2016-19","roc_auc"], g_auc],
                       "pr_auc":  [B1_lr.loc["ALL 2016-19","pr_auc"],  g_pr]},
                      index=["Logistic regression", "Gradient boosting"])
    display(B2)
else:
    print("Skipped (RUN_GBM = False)")

,roc_auc,pr_auc
Logistic regression,0.809,0.287
Gradient boosting,0.945,0.693


### B3. Do the drivers change over time?

In [11]:
c_past, p_past = fit_lr(past)
c_late, p_late = fit_lr(later)
both = pd.DataFrame({"FY2010-15": c_past, "FY2016-19": c_late}).dropna()
print(f"Spearman correlation of coefficients across eras: {both.corr(method='spearman').iloc[0,1]:.2f}")
top = both["FY2010-15"].abs().sort_values(ascending=False).head(12).index
t = both.loc[top].copy(); t["same_sign"] = np.sign(t["FY2010-15"]) == np.sign(t["FY2016-19"])
print(f"Term odds multiplier per +12 months:  FY2010-15 {term_odds_ratio_per_year(p_past, c_past):.3f} | FY2016-19 {term_odds_ratio_per_year(p_late, c_late):.3f}")
t

Spearman correlation of coefficients across eras: 0.72
Term odds multiplier per +12 months:  FY2010-15 0.742 | FY2016-19 0.644


,FY2010-15,FY2016-19,same_sign
num__TermInMonths,-1.907,-2.742,True
cat__ProcessingMethod_infrequent_sklearn,-1.774,-0.880,True
cat__BusinessAge_infrequent_sklearn,1.553,0.209,True
cat__ProcessingMethod_7aGeneral,0.904,0.949,True
cat__ProcessingMethod_WorkingCapitalCAPLine,-0.824,-1.522,True
cat__naics_sector_Finance_Insurance,-0.755,-0.659,True
num__log_gross_approval,0.665,0.575,True
cat__lender_type_Other,0.564,0.035,True
cat__ProjectState_HI,-0.520,-0.026,True
cat__lender_type_FDIC_bank,-0.518,-0.370,True


---
# Summary — what to take into the write-up
Fill in after running (numbers print above):

| Question | Result | Implication for the report |
|---|---|---|
| Is performance similar across samples? (A1) | | |
| Does longer term remain protective in FY2010–2019 and matured-only? (A2) | | |
| Are top drivers the same sign across samples? (A3) | | |
| What does it cost to predict forward in time? (B1) | | |
| Do the drivers shift between eras? (B3) | | |

**Limitations to state**
- Even FY2010–2019 is not fully resolved for 121+ month terms (~75%), so some residual selection bias remains.
- A time split with repeated `LocationID`s can still put the same location in both eras; the reported forward performance may be slightly optimistic.
- Logistic coefficients describe association, not cause.